In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pretrained ViT-B/16 Liver Tumor Classification Pipeline
========================================================
Google Colab compatible, PyTorch/Torchvision based.

Classes: Normal / Benign / Malignant

Includes:
- kaggle.json upload and automatic dataset download
- corrupted-image validation
- bilateral filtering + CLAHE preprocessing
- train-only augmentation
- ImageNet-1K pretrained ViT-B/16
- frozen-head training + last-3-transformer-block fine-tuning
- Accuracy, Precision, Recall/Sensitivity, Specificity, F1, AUC
- train/validation/test metrics, confusion matrices, ROC curves
- bright accuracy/loss curves and box plots
- model complexity and training time
- XAI for first 20 test images: saliency, 16x16 patch attribution, LIME
- results ZIP + automatic Colab download
"""

import os, sys, json, time, shutil, random, zipfile, warnings, subprocess, importlib.util
from pathlib import Path
from typing import List, Optional
warnings.filterwarnings("ignore")

def install_if_missing(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pkg in [
    ("kaggle", "kaggle"), ("cv2", "opencv-python-headless"),
    ("torch", "torch"), ("torchvision", "torchvision"),
    ("sklearn", "scikit-learn"), ("lime", "lime"),
    ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "Pillow")]:
    install_if_missing(imp, pkg)

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
from lime import lime_image
from skimage.segmentation import mark_boundaries

SEED=42
IMG_SIZE=224
PATCH_SIZE=16
PATCH_GRID=14
BATCH_SIZE=16
HEAD_EPOCHS=10
FINE_TUNE_EPOCHS=20
HEAD_LR=1e-3
FINE_TUNE_LR=2e-5
WEIGHT_DECAY=1e-4
TEST_SIZE=0.20
VAL_SIZE_FROM_REMAINING=0.20
N_XAI=20
LIME_NUM_SAMPLES=400

KAGGLE_KERNEL_REF="ahmedhamza1996/liver-tumor-classification"
KAGGLE_DATASET_SLUG=""

WORK=Path("/content/liver_vit_b16_pretrained_work") if Path("/content").exists() else Path.cwd()/"liver_vit_b16_pretrained_work"
DATA=WORK/"dataset"
RESULTS=WORK/"results_vit_b16_pretrained"
MODEL_DIR=RESULTS/"model"
METRICS=RESULTS/"metrics"
PLOTS=RESULTS/"plots"
XAI=RESULTS/"xai"
SAMPLES=RESULTS/"preprocessing_samples"
for p in [WORK,DATA,RESULTS,MODEL_DIR,METRICS,PLOTS,XAI,SAMPLES]: p.mkdir(parents=True,exist_ok=True)

CLASS_ALIASES={
    "normal":["normal","healthy","no_tumor","no-tumor","notumor"],
    "benign":["benign","cyst","hemangioma","hydatid"],
    "malignant":["malignant","cancer","hcc","metastasis","tumor"]}
CLASS_ORDER=["normal","benign","malignant"]
EXTS={".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:",torch.__version__); print("Device:",DEVICE)

# ---------- Kaggle ----------
def in_colab():
    try:
        import google.colab
        return True
    except Exception: return False

def configure_kaggle():
    kd=Path.home()/".kaggle"; kd.mkdir(parents=True,exist_ok=True); target=kd/"kaggle.json"
    if target.exists(): os.chmod(target,0o600); return
    for c in [Path.cwd()/"kaggle.json",Path("/content/kaggle.json"),WORK/"kaggle.json"]:
        if c.exists(): shutil.copy2(c,target); os.chmod(target,0o600); return
    if in_colab():
        from google.colab import files
        print("Please upload kaggle.json...")
        up=files.upload(); names=[x for x in up if x.lower().endswith(".json")]
        if not names: raise FileNotFoundError("No kaggle.json uploaded")
        shutil.copy2(Path(names[0]),target); os.chmod(target,0o600)
    else: raise FileNotFoundError("kaggle.json not found")

def cmd(c):
    print("[CMD]"," ".join(c))
    return subprocess.run(c,check=True,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)

def parse_sources(meta):
    out=[]
    for f in list(meta.rglob("*metadata*.json"))+list(meta.rglob("kernel-metadata.json")):
        try: d=json.loads(f.read_text())
        except Exception: continue
        for k in ["dataset_sources","datasetSources"]:
            for x in d.get(k,[]) if isinstance(d.get(k,[]),list) else []:
                if isinstance(x,str) and "/" in x: out.append(x)
                elif isinstance(x,dict):
                    r=x.get("ref") or x.get("source") or x.get("dataset")
                    if isinstance(r,str) and "/" in r: out.append(r)
    return sorted(set(out))

def discover_sources():
    meta=WORK/"kaggle_kernel_metadata"
    if meta.exists(): shutil.rmtree(meta)
    meta.mkdir(parents=True,exist_ok=True)
    for c in [["kaggle","kernels","pull",KAGGLE_KERNEL_REF,"-p",str(meta),"-m"],
              ["kaggle","kernels","pull","-p",str(meta),"-m",KAGGLE_KERNEL_REF]]:
        try:
            r=cmd(c); print(r.stdout[-1000:]); s=parse_sources(meta); print("Sources:",s); return s
        except Exception: pass
    return []

def download_data():
    configure_kaggle()
    slugs=[KAGGLE_DATASET_SLUG] if KAGGLE_DATASET_SLUG.strip() else discover_sources()
    if not slugs: raise RuntimeError("Could not detect dataset; set KAGGLE_DATASET_SLUG")
    for slug in slugs:
        dest=DATA/slug.replace("/","__"); dest.mkdir(parents=True,exist_ok=True); mark=dest/".done"
        if mark.exists() and any(dest.rglob("*")): continue
        r=cmd(["kaggle","datasets","download","-d",slug,"-p",str(dest),"--unzip"])
        print(r.stdout[-1000:]); mark.touch()
    (RESULTS/"kaggle_sources.txt").write_text("\n".join(slugs))
    return slugs

# ---------- Data ----------
def nt(x): return x.lower().replace(" ","_").replace("-","_")
def infer_label(p):
    parts=[nt(x) for x in p.parts]
    for lab in CLASS_ORDER:
        aliases=[nt(a) for a in CLASS_ALIASES[lab]]
        for part in reversed(parts[:-1]):
            if part==lab or part in aliases: return lab
    joined="/".join(parts)
    for lab in ["malignant","benign","normal"]:
        for a in CLASS_ALIASES[lab]:
            if nt(a) in joined: return lab
    return None

def read_bgr(path):
    try:
        raw=np.fromfile(path,dtype=np.uint8)
        return cv2.imdecode(raw,cv2.IMREAD_COLOR) if raw.size else None
    except Exception: return None

def read_rgb(path):
    b=read_bgr(path)
    if b is None: raise ValueError(path)
    return cv2.cvtColor(b,cv2.COLOR_BGR2RGB)

def collect_images():
    rows=[]; bad=[]
    for p in DATA.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in EXTS: continue
        lab=infer_label(p)
        if lab is None: continue
        im=read_bgr(str(p))
        if im is None or im.ndim!=3 or min(im.shape[:2])<8: bad.append({"filepath":str(p),"label":lab})
        else: rows.append({"filepath":str(p),"label":lab})
    pd.DataFrame(bad).to_csv(METRICS/"skipped_corrupt_images.csv",index=False)
    df=pd.DataFrame(rows).drop_duplicates("filepath").reset_index(drop=True)
    if df.empty: raise RuntimeError("No valid images found")
    print(df.label.value_counts()); return df

def split_df(df):
    tv,te=train_test_split(df,test_size=TEST_SIZE,random_state=SEED,stratify=df.label)
    tr,va=train_test_split(tv,test_size=VAL_SIZE_FROM_REMAINING,random_state=SEED,stratify=tv.label)
    for n,d in [("training",tr),("validation",va),("test",te)]:
        d.to_csv(METRICS/f"{n}_split.csv",index=False); print(n,len(d)); print(d.label.value_counts())
    return tr.reset_index(drop=True),va.reset_index(drop=True),te.reset_index(drop=True)

def preprocess(rgb):
    g=cv2.cvtColor(np.asarray(rgb,np.uint8),cv2.COLOR_RGB2GRAY)
    g=cv2.bilateralFilter(g,9,75,75)
    g=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(g)
    return cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)

def save_samples(df,n=9):
    s=df.sample(min(n,len(df)),random_state=SEED); fig,ax=plt.subplots(len(s),2,figsize=(8,max(6,3*len(s))))
    if len(s)==1: ax=np.array([ax])
    for i,(_,r) in enumerate(s.iterrows()):
        a=read_rgb(r.filepath); b=preprocess(a)
        ax[i,0].imshow(a); ax[i,0].set_title("Before: "+r.label); ax[i,0].axis("off")
        ax[i,1].imshow(b); ax[i,1].set_title("After: "+r.label); ax[i,1].axis("off")
    plt.tight_layout(); plt.savefig(SAMPLES/"before_after.png",dpi=300,bbox_inches="tight"); plt.close()

class LiverDataset(Dataset):
    def __init__(self,df,c2i,training=False):
        self.df=df.reset_index(drop=True); self.c2i=c2i
        aug=[transforms.Resize((IMG_SIZE,IMG_SIZE))]
        if training: aug += [transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(0.2),transforms.RandomRotation(20),transforms.RandomAffine(0,translate=(0.06,0.06),scale=(0.92,1.08))]
        aug += [transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
        self.tf=transforms.Compose(aug)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=Image.fromarray(preprocess(read_rgb(r.filepath)))
        return self.tf(im), self.c2i[r.label], r.filepath

# ---------- Model ----------
def build_model(nc):
    m=vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
    m.heads.head=nn.Linear(m.heads.head.in_features,nc)
    return m

def freeze_head(m):
    for p in m.parameters(): p.requires_grad=False
    for p in m.heads.parameters(): p.requires_grad=True

def unfreeze_last(m,n=3):
    for p in m.parameters(): p.requires_grad=False
    L=len(m.encoder.layers)
    for i in range(max(0,L-n),L):
        for p in m.encoder.layers[i].parameters(): p.requires_grad=True
    for p in m.encoder.ln.parameters(): p.requires_grad=True
    for p in m.heads.parameters(): p.requires_grad=True

def specificity(cm):
    vals=[]; total=cm.sum()
    for i in range(cm.shape[0]):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=total-tp-fn-fp
        vals.append(tn/(tn+fp) if tn+fp else np.nan)
    return np.asarray(vals)

def metrics(y,p,classes):
    pred=p.argmax(1); cm=confusion_matrix(y,pred,labels=np.arange(len(classes)))
    try: au=roc_auc_score(label_binarize(y,classes=np.arange(len(classes))),p,average="macro",multi_class="ovr")
    except Exception: au=np.nan
    rec=recall_score(y,pred,average="macro",zero_division=0)
    return dict(accuracy=accuracy_score(y,pred),precision_macro=precision_score(y,pred,average="macro",zero_division=0),recall_macro=rec,sensitivity_macro=rec,specificity_macro=float(np.nanmean(specificity(cm))),f1_macro=f1_score(y,pred,average="macro",zero_division=0),auc_macro_ovr=au)

def run_epoch(m,loader,crit,opt=None):
    train=opt is not None; m.train(train); total=0; ys=[]; ps=[]
    for x,y,_ in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        if train: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits=m(x); loss=crit(logits,y)
            if train: loss.backward(); opt.step()
        total += loss.item()*len(y); ys.append(y.detach().cpu().numpy()); ps.append(torch.softmax(logits.detach(),1).cpu().numpy())
    return total/len(loader.dataset),np.concatenate(ys),np.concatenate(ps)

def train_model(m,tr,va,te,classes,cw):
    crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float32,device=DEVICE))
    best=1e9; bestp=MODEL_DIR/"best_vit_b16.pth"; hist=[]; epoch=0
    stages=[("head",HEAD_EPOCHS,HEAD_LR,freeze_head),("finetune",FINE_TUNE_EPOCHS,FINE_TUNE_LR,lambda z:unfreeze_last(z,3))]
    for stage,epochs,lr,setup in stages:
        setup(m); opt=torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],lr=lr,weight_decay=WEIGHT_DECAY)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=0.2,patience=3); patience=0
        for _ in range(epochs):
            epoch+=1; tl,yt,pt=run_epoch(m,tr,crit,opt); vl,yv,pv=run_epoch(m,va,crit); tsl,ys,ps=run_epoch(m,te,crit)
            mt,mv,ms=metrics(yt,pt,classes),metrics(yv,pv,classes),metrics(ys,ps,classes); sch.step(vl)
            hist.append({"epoch":epoch,"stage":stage,"train_loss":tl,"val_loss":vl,"test_loss":tsl,**{f"train_{k}":v for k,v in mt.items()},**{f"val_{k}":v for k,v in mv.items()},**{f"test_{k}":v for k,v in ms.items()}})
            print(f"Epoch {epoch}: train={mt['accuracy']:.4f} val={mv['accuracy']:.4f} test={ms['accuracy']:.4f} val_loss={vl:.4f}")
            if vl<best: best=vl; patience=0; torch.save(m.state_dict(),bestp)
            else: patience+=1
            if patience>=8: break
    pd.DataFrame(hist).to_csv(METRICS/"training_history.csv",index=False)
    m.load_state_dict(torch.load(bestp,map_location=DEVICE,weights_only=True)); return pd.DataFrame(hist)

def evaluate(m,loader,name,classes):
    loss,y,p=run_epoch(m,loader,nn.CrossEntropyLoss()); pred=p.argmax(1); cm=confusion_matrix(y,pred,labels=np.arange(len(classes)))
    ov=metrics(y,p,classes); ov.update(split=name,loss=loss,n_samples=len(y)); pd.DataFrame([ov]).to_csv(METRICS/f"{name}_overall_metrics.csv",index=False)
    pr,re,f1,sup=precision_recall_fscore_support(y,pred,labels=np.arange(len(classes)),zero_division=0); sp=specificity(cm); yb=label_binarize(y,classes=np.arange(len(classes))); rows=[]
    for i,c in enumerate(classes):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        try: au=roc_auc_score(yb[:,i],p[:,i])
        except Exception: au=np.nan
        rows.append(dict(split=name,class_name=c,accuracy_ovr=(tp+tn)/cm.sum(),precision=pr[i],recall_sensitivity=re[i],specificity=sp[i],f1_score=f1[i],auc_ovr=au,support=sup[i]))
    pc=pd.DataFrame(rows); pc.to_csv(METRICS/f"{name}_per_class_metrics.csv",index=False)
    pd.DataFrame(classification_report(y,pred,labels=np.arange(len(classes)),target_names=classes,output_dict=True,zero_division=0)).T.to_csv(METRICS/f"{name}_classification_report.csv")
    return ov,y,pred,p,pc

# ---------- Plots ----------
def plot_history(h):
    plt.figure(figsize=(9,6)); plt.plot(h.epoch,h.train_accuracy,color="#00BFFF",label="Training",linewidth=2.5); plt.plot(h.epoch,h.val_accuracy,color="#FF1493",label="Validation",linewidth=2.5); plt.plot(h.epoch,h.test_accuracy,color="#32CD32",label="Test",linewidth=2.5); plt.ylim(0,1.05); plt.legend(); plt.grid(alpha=.25); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Pretrained ViT-B/16 Accuracy Curves"); plt.tight_layout(); plt.savefig(PLOTS/"accuracy_curves.png",dpi=300); plt.close()
    plt.figure(figsize=(9,6)); plt.plot(h.epoch,h.train_loss,color="#FF8C00",label="Training",linewidth=2.5); plt.plot(h.epoch,h.val_loss,color="#9400D3",label="Validation",linewidth=2.5); plt.plot(h.epoch,h.test_loss,color="#00CED1",label="Test",linewidth=2.5); plt.legend(); plt.grid(alpha=.25); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Pretrained ViT-B/16 Loss Curves"); plt.tight_layout(); plt.savefig(PLOTS/"loss_curves.png",dpi=300); plt.close()

def plot_eval(y,pred,prob,pc,classes,name):
    cm=confusion_matrix(y,pred,labels=np.arange(len(classes))); fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cm,cmap="turbo"); plt.colorbar(im,ax=ax); ax.set_xticks(range(len(classes)),classes,rotation=35); ax.set_yticks(range(len(classes)),classes); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name.title()+" Confusion Matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    plt.tight_layout(); plt.savefig(PLOTS/f"{name}_confusion_matrix.png",dpi=300); plt.close()
    yb=label_binarize(y,classes=np.arange(len(classes))); plt.figure(figsize=(8,7))
    for i,c in enumerate(classes):
        try: fpr,tpr,_=roc_curve(yb[:,i],prob[:,i]); plt.plot(fpr,tpr,label=f"{c} AUC={auc(fpr,tpr):.3f}",linewidth=2.2)
        except Exception: pass
    plt.plot([0,1],[0,1],"k--"); plt.legend(); plt.grid(alpha=.25); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(name.title()+" ROC-AUC"); plt.tight_layout(); plt.savefig(PLOTS/f"{name}_roc_auc.png",dpi=300); plt.close()
    cols=["accuracy_ovr","precision","recall_sensitivity","specificity","f1_score","auc_ovr"]; data=[pc[c].dropna().values for c in cols]; plt.figure(figsize=(10,6)); plt.boxplot(data,labels=["Accuracy","Precision","Recall","Specificity","F1","AUC"],patch_artist=True,showmeans=True); plt.ylim(0,1.05); plt.grid(axis="y",alpha=.25); plt.title(name.title()+" Metric Box Plot"); plt.tight_layout(); plt.savefig(PLOTS/f"{name}_metric_boxplot.png",dpi=300); plt.close()

# ---------- XAI ----------
def xai_tensor(rgb):
    tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    return tf(Image.fromarray(preprocess(rgb)))

def normmap(x):
    x=np.nan_to_num(np.asarray(x,np.float32)); x-=x.min(); return x/(x.max()+1e-8)

def overlay(rgb,h):
    h=cv2.resize(normmap(h),(rgb.shape[1],rgb.shape[0])); c=cv2.cvtColor(cv2.applyColorMap(np.uint8(h*255),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB); return cv2.addWeighted(rgb.astype(np.uint8),.55,c,.45,0)

def saliency_patch(m,t,pred):
    x=t.unsqueeze(0).to(DEVICE); x.requires_grad_(True); m.eval(); m.zero_grad(set_to_none=True); s=m(x)[0,pred]; s.backward(); g=x.grad.detach()[0]; sal=g.abs().max(0).values.cpu().numpy(); ge=g.abs().mean(0,keepdim=True).unsqueeze(0); patch=torch.nn.functional.avg_pool2d(ge,PATCH_SIZE,PATCH_SIZE).squeeze().cpu().numpy(); return normmap(sal),normmap(patch)

def generate_xai(m,test_df,classes,c2i):
    for d in ["saliency","patch_attribution","lime","combined"]: (XAI/d).mkdir(parents=True,exist_ok=True)
    def predict(images):
        batch=torch.stack([xai_tensor(np.asarray(im,np.uint8)) for im in images]).to(DEVICE)
        with torch.no_grad(): return torch.softmax(m(batch),1).cpu().numpy()
    expl=lime_image.LimeImageExplainer(random_state=SEED); rows=[]
    for k,(_,r) in enumerate(test_df.head(N_XAI).iterrows(),1):
        print(f"XAI {k}/{min(N_XAI,len(test_df))}: {r.filepath}"); rgb=cv2.resize(read_rgb(r.filepath),(IMG_SIZE,IMG_SIZE)); t=xai_tensor(rgb)
        with torch.no_grad(): prob=torch.softmax(m(t.unsqueeze(0).to(DEVICE)),1)[0].cpu().numpy()
        pred=int(prob.argmax()); true=c2i[r.label]; sal,patch=saliency_patch(m,t,pred); salim=overlay(rgb,sal); patchim=overlay(rgb,patch); stem=f"Patient_{k:02d}_true_{classes[true]}_pred_{classes[pred]}"
        cv2.imwrite(str(XAI/"saliency"/f"{stem}.png"),cv2.cvtColor(salim,cv2.COLOR_RGB2BGR)); cv2.imwrite(str(XAI/"patch_attribution"/f"{stem}.png"),cv2.cvtColor(patchim,cv2.COLOR_RGB2BGR)); limeim=rgb.copy()
        try:
            e=expl.explain_instance(rgb.astype(float),predict,top_labels=len(classes),hide_color=0,num_samples=LIME_NUM_SAMPLES,random_seed=SEED); temp,mask=e.get_image_and_mask(pred,positive_only=True,num_features=10,hide_rest=False); limeim=np.uint8(np.clip(mark_boundaries(np.clip(temp,0,255)/255.,mask),0,1)*255); cv2.imwrite(str(XAI/"lime"/f"{stem}.png"),cv2.cvtColor(limeim,cv2.COLOR_RGB2BGR))
        except Exception as ex: print("LIME failed:",ex)
        fig,ax=plt.subplots(1,4,figsize=(18,5))
        for a,im,title in zip(ax,[rgb,salim,patchim,limeim],["Original","Saliency","16x16 Patch Attribution","LIME"]): a.imshow(im); a.set_title(title); a.axis("off")
        fig.suptitle(f"True: {classes[true]} | Pred: {classes[pred]} ({prob[pred]:.4f})"); plt.tight_layout(); plt.savefig(XAI/"combined"/f"{stem}.png",dpi=250); plt.close(); rec={"patient_number":k,"filepath":r.filepath,"true_class":classes[true],"predicted_class":classes[pred],"confidence":float(prob[pred]),"correct":int(true==pred)}; rec.update({f"prob_{c}":float(prob[i]) for i,c in enumerate(classes)}); rows.append(rec)
    pd.DataFrame(rows).to_csv(XAI/"first_20_test_xai_predictions.csv",index=False)

# ---------- Main ----------
def main():
    start=time.perf_counter(); slugs=download_data(); df=collect_images(); classes=[c for c in CLASS_ORDER if c in df.label.unique()]; c2i={c:i for i,c in enumerate(classes)}; (RESULTS/"class_mapping.json").write_text(json.dumps(c2i,indent=2)); tr,va,te=split_df(df); save_samples(tr)
    dtr=LiverDataset(tr,c2i,True); dtre=LiverDataset(tr,c2i,False); dva=LiverDataset(va,c2i,False); dte=LiverDataset(te,c2i,False)
    kwargs=dict(batch_size=BATCH_SIZE,num_workers=2,pin_memory=DEVICE.type=="cuda"); ltr=DataLoader(dtr,shuffle=True,**kwargs); ltre=DataLoader(dtre,shuffle=False,**kwargs); lva=DataLoader(dva,shuffle=False,**kwargs); lte=DataLoader(dte,shuffle=False,**kwargs)
    y0=np.array([c2i[x] for x in tr.label]); cw=compute_class_weight(class_weight="balanced",classes=np.arange(len(classes)),y=y0); print("Class weights:",cw)
    print("Loading torchvision ImageNet-1K pretrained ViT-B/16..."); m=build_model(len(classes)).to(DEVICE); print("Parameters:",sum(p.numel() for p in m.parameters()))
    ts=time.perf_counter(); h=train_model(m,ltr,lva,lte,classes,cw); training_time=time.perf_counter()-ts; plot_history(h); allm=[]
    for name,loader in [("training",ltre),("validation",lva),("test",lte)]:
        ov,y,pred,prob,pc=evaluate(m,loader,name,classes); allm.append(ov); plot_eval(y,pred,prob,pc,classes,name)
    pd.DataFrame(allm).to_csv(METRICS/"all_split_overall_metrics.csv",index=False); print(pd.DataFrame(allm).to_string(index=False))
    total=sum(p.numel() for p in m.parameters()); trainable=sum(p.numel() for p in m.parameters() if p.requires_grad); comp={"model":"Pretrained ViT-B/16","weights":"ViT_B_16_Weights.IMAGENET1K_V1","input_size":224,"patch_size":16,"patch_grid":"14x14","total_parameters":total,"trainable_parameters_after_finetuning":trainable,"non_trainable_parameters_after_finetuning":total-trainable,"reference_gflops":17.56,"training_time_seconds":training_time,"training_time_minutes":training_time/60}; pd.DataFrame([comp]).to_csv(METRICS/"model_complexity_and_time.csv",index=False); (METRICS/"model_complexity_and_time.json").write_text(json.dumps(comp,indent=2)); torch.save({"model_state_dict":m.state_dict(),"classes":classes,"class_to_idx":c2i,"architecture":"vit_b_16","weights":"IMAGENET1K_V1"},MODEL_DIR/"final_pretrained_vit_b16.pth")
    generate_xai(m,te,classes,c2i); meta={"model":"Pretrained ViT-B/16","framework":"PyTorch/Torchvision","weights":"IMAGENET1K_V1","classes":classes,"kaggle_sources":slugs,"total_pipeline_seconds":time.perf_counter()-start}; (RESULTS/"run_metadata.json").write_text(json.dumps(meta,indent=2))
    z=WORK/"Pretrained_ViT_B16_Liver_Tumor_Results.zip"; z.unlink(missing_ok=True)
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as f:
        for p in RESULTS.rglob("*"):
            if p.is_file(): f.write(p,arcname=p.relative_to(RESULTS.parent))
    print("ZIP:",z)
    if in_colab():
        from google.colab import files
        files.download(str(z))

if __name__=="__main__": main()


[INSTALL] lime
PyTorch: 2.11.0+cu128
Device: cuda
Please upload kaggle.json...


Saving kaggle.json to kaggle.json
[CMD] kaggle kernels pull ahmedhamza1996/liver-tumor-classification -p /content/liver_vit_b16_pretrained_work/kaggle_kernel_metadata -m
Source code and metadata downloaded to /content/liver_vit_b16_pretrained_work/kaggle_kernel_metadata

Sources: ['ahmedhamza1996/dataset']
[CMD] kaggle datasets download -d ahmedhamza1996/dataset -p /content/liver_vit_b16_pretrained_work/dataset/ahmedhamza1996__dataset --unzip
hamza1996/dataset
License(s): unknown

  0%|          | 0.00/241M [00:00<?, ?B/s]
  3%|▎         | 7.00M/241M [00:00<00:04, 59.5MB/s]
  5%|▌         | 13.0M/241M [00:00<00:04, 58.7MB/s]
 11%|█         | 26.0M/241M [00:00<00:02, 91.1MB/s]
 15%|█▍        | 35.0M/241M [00:00<00:05, 38.2MB/s]
 21%|██        | 51.0M/241M [00:00<00:03, 60.0MB/s]
 25%|██▍       | 60.0M/241M [00:01<00:02, 64.0MB/s]
 34%|███▎      | 81.0M/241M [00:01<00:02, 80.2MB/s]
 41%|████      | 99.0M/241M [00:01<00:01, 101MB/s] 
 50%|█████     | 121M/241M [00:01<00:01, 123MB/s] 
 56%

100%|██████████| 330M/330M [00:01<00:00, 176MB/s]


Parameters: 85800963
Epoch 1: train=0.6638 val=0.8151 test=0.8736 val_loss=0.4590
Epoch 2: train=0.8914 val=0.9452 test=0.9505 val_loss=0.2905
Epoch 3: train=0.9379 val=0.9589 test=0.9396 val_loss=0.2431
Epoch 4: train=0.9466 val=0.9658 test=0.9560 val_loss=0.1791
Epoch 5: train=0.9621 val=0.9726 test=0.9615 val_loss=0.1505
Epoch 6: train=0.9672 val=0.9795 test=0.9670 val_loss=0.1305
Epoch 7: train=0.9759 val=0.9658 test=0.9560 val_loss=0.1222
Epoch 8: train=0.9862 val=0.9863 test=0.9725 val_loss=0.1035
Epoch 9: train=0.9793 val=0.9863 test=0.9780 val_loss=0.0981
Epoch 10: train=0.9879 val=0.9932 test=0.9835 val_loss=0.0838
Epoch 11: train=0.9724 val=0.9795 test=0.9890 val_loss=0.0877
Epoch 12: train=0.9966 val=1.0000 test=0.9945 val_loss=0.0157
Epoch 13: train=0.9966 val=1.0000 test=0.9945 val_loss=0.0135
Epoch 14: train=0.9897 val=1.0000 test=1.0000 val_loss=0.0074
Epoch 15: train=0.9810 val=1.0000 test=0.9890 val_loss=0.0080
Epoch 16: train=1.0000 val=1.0000 test=1.0000 val_loss=0.0

  0%|          | 0/400 [00:00<?, ?it/s]

...

XAI 20/20: /content/liver_vit_b16_pretrained_work/dataset/ahmedhamza1996__dataset/output/malignant/(55).PNG


  0%|          | 0/400 [00:00<?, ?it/s]

ZIP: /content/liver_vit_b16_pretrained_work/Pretrained_ViT_B16_Liver_Tumor_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

file_path = "/content/liver_vit_b16_pretrained_work/Pretrained_ViT_B16_Liver_Tumor_Results.zip"

files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Swin Transformer

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pretrained Swin Transformer Tiny Liver Tumor Classification Pipeline
====================================================================

Google Colab compatible, PyTorch/Torchvision based.

Classes:
    Normal / Benign / Malignant

Pipeline:
    kaggle.json upload
    -> automatic Kaggle dataset discovery/download
    -> corrupted image validation
    -> preprocessing (bilateral filter + CLAHE)
    -> train-only augmentation
    -> pretrained Swin Transformer Tiny (ImageNet-1K)
    -> frozen classifier-head training
    -> fine-tuning final Swin stage
    -> train/validation/test metrics
    -> confusion matrices + ROC curves
    -> bright accuracy/loss curves
    -> metric/confidence box plots
    -> model complexity + training time
    -> XAI for first 20 test images:
       * input-gradient saliency
       * feature-map gradient attribution
       * LIME
    -> ZIP + automatic download

Official pretrained model:
    torchvision.models.swin_t(
        weights=Swin_T_Weights.IMAGENET1K_V1
    )
"""

# ============================================================
# 0. DEPENDENCIES
# ============================================================
import os
import sys
import json
import time
import shutil
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path
from typing import List, Optional

warnings.filterwarnings("ignore")

def install_if_missing(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pkg in [
    ("kaggle", "kaggle"),
    ("cv2", "opencv-python-headless"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("sklearn", "scikit-learn"),
    ("lime", "lime"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("PIL", "Pillow"),
]:
    install_if_missing(imp, pkg)

# ============================================================
# 1. IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.utils.class_weight import compute_class_weight

from lime import lime_image
from skimage.segmentation import mark_boundaries

# ============================================================
# 2. CONFIG
# ============================================================
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16

HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 20
TOTAL_EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS

HEAD_LR = 1e-3
FINE_TUNE_LR = 2e-5
WEIGHT_DECAY = 1e-4

TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAINING = 0.20

KAGGLE_KERNEL_REF = "ahmedhamza1996/liver-tumor-classification"
KAGGLE_DATASET_SLUG = ""

USE_BILATERAL_FILTER = True
USE_CLAHE = True

N_XAI = 20
LIME_NUM_SAMPLES = 400

WORK_DIR = Path("/content/liver_swin_tiny_work") if Path("/content").exists() else Path.cwd() / "liver_swin_tiny_work"
DATA_DIR = WORK_DIR / "dataset"
RESULTS_DIR = WORK_DIR / "results_swin_tiny"
MODEL_DIR = RESULTS_DIR / "model"
METRICS_DIR = RESULTS_DIR / "metrics"
PLOTS_DIR = RESULTS_DIR / "plots"
XAI_DIR = RESULTS_DIR / "xai"
SAMPLES_DIR = RESULTS_DIR / "preprocessing_samples"

for p in [WORK_DIR, DATA_DIR, RESULTS_DIR, MODEL_DIR, METRICS_DIR, PLOTS_DIR, XAI_DIR, SAMPLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CLASS_ALIASES = {
    "normal": ["normal", "healthy", "no_tumor", "no-tumor", "notumor"],
    "benign": ["benign", "cyst", "hemangioma", "hydatid"],
    "malignant": ["malignant", "cancer", "hcc", "metastasis", "tumor"],
}
TARGET_CLASS_ORDER = ["normal", "benign", "malignant"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

BRIGHT = {
    "train": "#00BFFF",
    "val": "#FF1493",
    "test": "#32CD32",
    "train_loss": "#FF8C00",
    "val_loss": "#9400D3",
    "test_loss": "#00CED1",
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ============================================================
# 3. KAGGLE AUTHENTICATION + DOWNLOAD
# ============================================================
def in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

def configure_kaggle():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    if target.exists():
        os.chmod(target, 0o600)
        print("Kaggle credentials found:", target)
        return

    candidates = [
        Path.cwd() / "kaggle.json",
        Path("/content/kaggle.json"),
        WORK_DIR / "kaggle.json",
    ]

    for c in candidates:
        if c.exists():
            shutil.copy2(c, target)
            os.chmod(target, 0o600)
            print("Kaggle credentials configured from:", c)
            return

    if in_colab():
        from google.colab import files
        print("\nPlease upload kaggle.json...")
        uploaded = files.upload()
        json_files = [name for name in uploaded if name.lower().endswith(".json")]
        if not json_files:
            raise FileNotFoundError("No kaggle.json uploaded.")
        shutil.copy2(Path(json_files[0]), target)
        os.chmod(target, 0o600)
        print("Kaggle credentials configured.")
    else:
        raise FileNotFoundError("kaggle.json not found.")

def run_cmd(cmd):
    print("[CMD]", " ".join(cmd))
    return subprocess.run(
        cmd,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

def parse_kernel_sources(metadata_dir: Path) -> List[str]:
    slugs = []

    metadata_files = (
        list(metadata_dir.rglob("*metadata*.json"))
        + list(metadata_dir.rglob("kernel-metadata.json"))
    )

    for mf in metadata_files:
        try:
            data = json.loads(mf.read_text(encoding="utf-8"))
        except Exception:
            continue

        for key in ["dataset_sources", "datasetSources"]:
            vals = data.get(key, [])
            if isinstance(vals, list):
                for item in vals:
                    if isinstance(item, str) and "/" in item:
                        slugs.append(item)
                    elif isinstance(item, dict):
                        ref = item.get("ref") or item.get("source") or item.get("dataset")
                        if isinstance(ref, str) and "/" in ref:
                            slugs.append(ref)

        vals = data.get("data_sources", [])
        if isinstance(vals, list):
            for item in vals:
                if isinstance(item, dict):
                    ref = item.get("ref") or item.get("source")
                    if isinstance(ref, str) and "/" in ref:
                        slugs.append(ref)

    return sorted(set(slugs))

def discover_kernel_inputs():
    meta_dir = WORK_DIR / "kaggle_kernel_metadata"

    if meta_dir.exists():
        shutil.rmtree(meta_dir)

    meta_dir.mkdir(parents=True, exist_ok=True)

    commands = [
        ["kaggle", "kernels", "pull", KAGGLE_KERNEL_REF, "-p", str(meta_dir), "-m"],
        ["kaggle", "kernels", "pull", "-p", str(meta_dir), "-m", KAGGLE_KERNEL_REF],
    ]

    for cmd in commands:
        try:
            result = run_cmd(cmd)
            print(result.stdout[-1200:])
            slugs = parse_kernel_sources(meta_dir)
            print("Discovered Kaggle dataset sources:", slugs)
            return slugs
        except Exception:
            continue

    return []

def download_data():
    configure_kaggle()

    slugs = (
        [KAGGLE_DATASET_SLUG.strip()]
        if KAGGLE_DATASET_SLUG.strip()
        else discover_kernel_inputs()
    )

    if not slugs:
        raise RuntimeError(
            "No Kaggle dataset slug detected. Set KAGGLE_DATASET_SLUG manually."
        )

    for slug in slugs:
        dest = DATA_DIR / slug.replace("/", "__")
        dest.mkdir(parents=True, exist_ok=True)

        marker = dest / ".download_complete"

        if marker.exists() and any(dest.rglob("*")):
            print("Dataset already downloaded:", slug)
            continue

        result = run_cmd([
            "kaggle", "datasets", "download",
            "-d", slug,
            "-p", str(dest),
            "--unzip",
        ])

        print(result.stdout[-1200:])
        marker.touch()

    (RESULTS_DIR / "kaggle_sources.txt").write_text(
        "\n".join(slugs),
        encoding="utf-8"
    )

    return slugs

# ============================================================
# 4. DATASET DISCOVERY
# ============================================================
def norm_token(text):
    return text.lower().replace(" ", "_").replace("-", "_")

def infer_label(path: Path) -> Optional[str]:
    parts = [norm_token(p) for p in path.parts]

    for label in TARGET_CLASS_ORDER:
        aliases = [norm_token(x) for x in CLASS_ALIASES[label]]
        for part in reversed(parts[:-1]):
            if part == label or part in aliases:
                return label

    joined = "/".join(parts)

    for label in ["malignant", "benign", "normal"]:
        for alias in CLASS_ALIASES[label]:
            if norm_token(alias) in joined:
                return label

    return None

def robust_read_bgr(path):
    try:
        raw = np.fromfile(path, dtype=np.uint8)
        if raw.size == 0:
            return None
        return cv2.imdecode(raw, cv2.IMREAD_COLOR)
    except Exception:
        return None

def collect_images():
    rows = []
    bad = []

    for p in DATA_DIR.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        label = infer_label(p)
        if label is None:
            continue

        img = robust_read_bgr(str(p))

        if img is None or img.ndim != 3 or min(img.shape[:2]) < 8:
            bad.append({
                "filepath": str(p),
                "label": label
            })
        else:
            rows.append({
                "filepath": str(p),
                "label": label
            })

    pd.DataFrame(bad).to_csv(
        METRICS_DIR / "skipped_corrupt_images.csv",
        index=False
    )

    df = (
        pd.DataFrame(rows)
        .drop_duplicates("filepath")
        .reset_index(drop=True)
    )

    if df.empty:
        raise RuntimeError("No valid images found.")

    print("\nDetected classes:")
    print(df["label"].value_counts())

    return df

def create_splits(df):
    train_val, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        stratify=df["label"],
        random_state=SEED
    )

    train_df, val_df = train_test_split(
        train_val,
        test_size=VAL_SIZE_FROM_REMAINING,
        stratify=train_val["label"],
        random_state=SEED
    )

    for name, split in [
        ("training", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        split.to_csv(
            METRICS_DIR / f"{name}_split.csv",
            index=False
        )

        print(f"\n{name.upper()} ({len(split)})")
        print(split["label"].value_counts())

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

# ============================================================
# 5. PREPROCESSING
# ============================================================
def read_rgb(path):
    bgr = robust_read_bgr(path)

    if bgr is None:
        raise ValueError(f"Cannot decode image: {path}")

    return cv2.cvtColor(
        bgr,
        cv2.COLOR_BGR2RGB
    )

def paper_preprocess(image_rgb):
    image_rgb = np.asarray(
        image_rgb,
        dtype=np.uint8
    )

    gray = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2GRAY
    )

    if USE_BILATERAL_FILTER:
        gray = cv2.bilateralFilter(
            gray,
            9,
            75,
            75
        )

    if USE_CLAHE:
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )
        gray = clahe.apply(gray)

    return cv2.cvtColor(
        gray,
        cv2.COLOR_GRAY2RGB
    )

def save_before_after(df, n=9):
    sample = df.sample(
        min(n, len(df)),
        random_state=SEED
    )

    fig, axes = plt.subplots(
        len(sample),
        2,
        figsize=(8, max(3 * len(sample), 6))
    )

    if len(sample) == 1:
        axes = np.array([axes])

    for i, (_, row) in enumerate(sample.iterrows()):
        original = read_rgb(row.filepath)
        processed = paper_preprocess(original)

        axes[i, 0].imshow(original)
        axes[i, 0].set_title(
            f"Before: {row.label}",
            fontweight="bold"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(processed)
        axes[i, 1].set_title(
            f"After: {row.label}",
            fontweight="bold"
        )
        axes[i, 1].axis("off")

    plt.tight_layout()

    plt.savefig(
        SAMPLES_DIR / "before_after_preprocessing.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# ============================================================
# 6. DATASET + TRANSFORMS
# ============================================================
class LiverDataset(Dataset):
    def __init__(
        self,
        df,
        class_to_idx,
        training=False
    ):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx

        if training:
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.20),
                transforms.RandomRotation(20),
                transforms.RandomAffine(
                    degrees=0,
                    translate=(0.06, 0.06),
                    scale=(0.92, 1.08)
                ),
                transforms.ColorJitter(
                    brightness=0.10,
                    contrast=0.10
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    IMAGENET_MEAN,
                    IMAGENET_STD
                ),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(
                    IMAGENET_MEAN,
                    IMAGENET_STD
                ),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb = read_rgb(row.filepath)
        rgb = paper_preprocess(rgb)

        image = self.transform(
            Image.fromarray(rgb)
        )

        label = self.class_to_idx[
            row.label
        ]

        return image, label, row.filepath

# ============================================================
# 7. MODEL
# ============================================================
def build_model(num_classes):
    weights = Swin_T_Weights.IMAGENET1K_V1

    model = swin_t(
        weights=weights
    )

    in_features = model.head.in_features

    model.head = nn.Linear(
        in_features,
        num_classes
    )

    return model

def freeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = False

    for p in model.head.parameters():
        p.requires_grad = True

def unfreeze_last_stage(model):
    for p in model.parameters():
        p.requires_grad = False

    # Swin-T features:
    # patch embedding + stages + patch merging.
    # Fine-tune the final Swin stage and final normalization.
    if hasattr(model, "features"):
        total = len(model.features)

        for idx in range(max(0, total - 2), total):
            for p in model.features[idx].parameters():
                p.requires_grad = True

    if hasattr(model, "norm"):
        for p in model.norm.parameters():
            p.requires_grad = True

    for p in model.head.parameters():
        p.requires_grad = True

def trainable_parameters(model):
    return [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

# ============================================================
# 8. METRICS
# ============================================================
def specificity_per_class(cm):
    total = cm.sum()
    vals = []

    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp

        vals.append(
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

    return np.asarray(
        vals,
        dtype=float
    )

def compute_metrics(
    y_true,
    probs,
    classes
):
    pred = np.argmax(
        probs,
        axis=1
    )

    cm = confusion_matrix(
        y_true,
        pred,
        labels=np.arange(len(classes))
    )

    precision = precision_score(
        y_true,
        pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        pred,
        average="macro",
        zero_division=0
    )

    specificity = float(
        np.nanmean(
            specificity_per_class(cm)
        )
    )

    try:
        yb = label_binarize(
            y_true,
            classes=np.arange(len(classes))
        )

        auc_macro = roc_auc_score(
            yb,
            probs,
            average="macro",
            multi_class="ovr"
        )
    except Exception:
        auc_macro = np.nan

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                pred
            )
        ),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "sensitivity_macro": float(recall),
        "specificity_macro": float(specificity),
        "f1_macro": float(f1),
        "auc_macro_ovr": float(auc_macro),
    }

# ============================================================
# 9. TRAINING
# ============================================================
def run_loader(
    model,
    loader,
    criterion,
    optimizer=None
):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    labels_all = []
    probs_all = []

    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

    for images, labels, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(
                enabled=(DEVICE.type == "cuda")
            ):
                logits = model(images)
                loss = criterion(
                    logits,
                    labels
                )

            if training:
                scaler.scale(
                    loss
                ).backward()

                scaler.step(
                    optimizer
                )

                scaler.update()

        probs = torch.softmax(
            logits.detach(),
            dim=1
        )

        total_loss += (
            loss.item()
            * len(labels)
        )

        labels_all.append(
            labels.detach()
            .cpu()
            .numpy()
        )

        probs_all.append(
            probs.cpu()
            .numpy()
        )

    return (
        total_loss
        / len(loader.dataset),
        np.concatenate(labels_all),
        np.concatenate(probs_all)
    )

def fit_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    classes,
    class_weights
):
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=DEVICE
        )
    )

    history = []

    best_val_loss = float("inf")
    best_path = MODEL_DIR / "best_swin_tiny.pth"

    patience = 8
    patience_counter = 0

    freeze_backbone(model)

    optimizer = torch.optim.AdamW(
        trainable_parameters(model),
        lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.2,
        patience=3
    )

    global_epoch = 0

    # ---------------- HEAD TRAINING ----------------
    for _ in range(HEAD_EPOCHS):
        global_epoch += 1

        train_loss, ytr, ptr = run_loader(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_loss, yv, pv = run_loader(
            model,
            val_loader,
            criterion
        )

        test_loss, yt, pt = run_loader(
            model,
            test_loader,
            criterion
        )

        train_metrics = compute_metrics(
            ytr,
            ptr,
            classes
        )

        val_metrics = compute_metrics(
            yv,
            pv,
            classes
        )

        test_metrics = compute_metrics(
            yt,
            pt,
            classes
        )

        scheduler.step(
            val_loss
        )

        row = {
            "epoch": global_epoch,
            "stage": "head_training",
            "train_loss": train_loss,
            "val_loss": val_loss,
            "test_loss": test_loss,
            **{
                f"train_{k}": v
                for k, v
                in train_metrics.items()
            },
            **{
                f"val_{k}": v
                for k, v
                in val_metrics.items()
            },
            **{
                f"test_{k}": v
                for k, v
                in test_metrics.items()
            }
        }

        history.append(row)

        print(
            f"Epoch {global_epoch:02d}/{TOTAL_EPOCHS} | "
            f"train_acc={train_metrics['accuracy']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"test_acc={test_metrics['accuracy']:.4f} | "
            f"val_loss={val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                best_path
            )
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(
                "Early stopping during head training."
            )
            break

    # ---------------- FINE TUNING ----------------
    unfreeze_last_stage(model)

    optimizer = torch.optim.AdamW(
        trainable_parameters(model),
        lr=FINE_TUNE_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.2,
        patience=3
    )

    patience_counter = 0

    for _ in range(FINE_TUNE_EPOCHS):
        global_epoch += 1

        train_loss, ytr, ptr = run_loader(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_loss, yv, pv = run_loader(
            model,
            val_loader,
            criterion
        )

        test_loss, yt, pt = run_loader(
            model,
            test_loader,
            criterion
        )

        train_metrics = compute_metrics(
            ytr,
            ptr,
            classes
        )

        val_metrics = compute_metrics(
            yv,
            pv,
            classes
        )

        test_metrics = compute_metrics(
            yt,
            pt,
            classes
        )

        scheduler.step(
            val_loss
        )

        row = {
            "epoch": global_epoch,
            "stage": "fine_tuning",
            "train_loss": train_loss,
            "val_loss": val_loss,
            "test_loss": test_loss,
            **{
                f"train_{k}": v
                for k, v
                in train_metrics.items()
            },
            **{
                f"val_{k}": v
                for k, v
                in val_metrics.items()
            },
            **{
                f"test_{k}": v
                for k, v
                in test_metrics.items()
            }
        }

        history.append(row)

        print(
            f"Epoch {global_epoch:02d}/{TOTAL_EPOCHS} | "
            f"train_acc={train_metrics['accuracy']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"test_acc={test_metrics['accuracy']:.4f} | "
            f"val_loss={val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                best_path
            )
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(
                "Early stopping during fine-tuning."
            )
            break

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        METRICS_DIR / "training_history.csv",
        index=False
    )

    model.load_state_dict(
        torch.load(
            best_path,
            map_location=DEVICE,
            weights_only=True
        )
    )

    return history_df

# ============================================================
# 10. FINAL EVALUATION
# ============================================================
def evaluate_split(
    model,
    loader,
    split_name,
    classes
):
    criterion = nn.CrossEntropyLoss()

    loss, y_true, probs = run_loader(
        model,
        loader,
        criterion
    )

    pred = np.argmax(
        probs,
        axis=1
    )

    cm = confusion_matrix(
        y_true,
        pred,
        labels=np.arange(len(classes))
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        pred,
        labels=np.arange(len(classes)),
        zero_division=0
    )

    specificity = specificity_per_class(
        cm
    )

    overall = compute_metrics(
        y_true,
        probs,
        classes
    )

    overall.update({
        "split": split_name,
        "loss": float(loss),
        "n_samples": int(len(y_true))
    })

    pd.DataFrame(
        [overall]
    ).to_csv(
        METRICS_DIR / f"{split_name}_overall_metrics.csv",
        index=False
    )

    yb = label_binarize(
        y_true,
        classes=np.arange(len(classes))
    )

    rows = []

    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        try:
            auc_i = roc_auc_score(
                yb[:, i],
                probs[:, i]
            )
        except Exception:
            auc_i = np.nan

        rows.append({
            "split": split_name,
            "class": cls,
            "accuracy_ovr": (
                (tp + tn)
                / cm.sum()
            ),
            "precision": precision[i],
            "recall_sensitivity": recall[i],
            "specificity": specificity[i],
            "f1_score": f1[i],
            "auc_ovr": auc_i,
            "support": support[i],
        })

    per_class_df = pd.DataFrame(
        rows
    )

    per_class_df.to_csv(
        METRICS_DIR / f"{split_name}_per_class_metrics.csv",
        index=False
    )

    pd.DataFrame(
        classification_report(
            y_true,
            pred,
            labels=np.arange(len(classes)),
            target_names=classes,
            output_dict=True,
            zero_division=0
        )
    ).T.to_csv(
        METRICS_DIR / f"{split_name}_classification_report.csv"
    )

    pred_df = pd.DataFrame({
        "true_index": y_true,
        "predicted_index": pred,
        "true_class": [
            classes[i]
            for i in y_true
        ],
        "predicted_class": [
            classes[i]
            for i in pred
        ],
        "confidence": probs.max(axis=1),
        "correct": (
            y_true == pred
        ).astype(int),
    })

    for i, cls in enumerate(classes):
        pred_df[
            f"prob_{cls}"
        ] = probs[:, i]

    pred_df.to_csv(
        METRICS_DIR / f"{split_name}_predictions.csv",
        index=False
    )

    return (
        overall,
        y_true,
        pred,
        probs,
        per_class_df
    )

# ============================================================
# 11. PLOTS
# ============================================================
def plot_history(history):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.plot(
        history.epoch,
        history.train_accuracy,
        color=BRIGHT["train"],
        linewidth=2.5,
        label="Training"
    )

    ax.plot(
        history.epoch,
        history.val_accuracy,
        color=BRIGHT["val"],
        linewidth=2.5,
        label="Validation"
    )

    ax.plot(
        history.epoch,
        history.test_accuracy,
        color=BRIGHT["test"],
        linewidth=2.5,
        label="Test"
    )

    ax.set_xlabel(
        "Epoch",
        fontweight="bold"
    )

    ax.set_ylabel(
        "Accuracy",
        fontweight="bold"
    )

    ax.set_ylim(
        0,
        1.05
    )

    ax.set_title(
        "Pretrained Swin-T Accuracy Curves",
        fontweight="bold"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / "accuracy_curves.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.plot(
        history.epoch,
        history.train_loss,
        color=BRIGHT["train_loss"],
        linewidth=2.5,
        label="Training"
    )

    ax.plot(
        history.epoch,
        history.val_loss,
        color=BRIGHT["val_loss"],
        linewidth=2.5,
        label="Validation"
    )

    ax.plot(
        history.epoch,
        history.test_loss,
        color=BRIGHT["test_loss"],
        linewidth=2.5,
        label="Test"
    )

    ax.set_xlabel(
        "Epoch",
        fontweight="bold"
    )

    ax.set_ylabel(
        "Loss",
        fontweight="bold"
    )

    ax.set_title(
        "Pretrained Swin-T Loss Curves",
        fontweight="bold"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / "loss_curves.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

def plot_confusion(
    y_true,
    pred,
    classes,
    split
):
    cm = confusion_matrix(
        y_true,
        pred,
        labels=np.arange(len(classes))
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    im = ax.imshow(
        cm,
        cmap="turbo"
    )

    plt.colorbar(
        im,
        ax=ax
    )

    ax.set_xticks(
        range(len(classes)),
        classes,
        rotation=35,
        ha="right"
    )

    ax.set_yticks(
        range(len(classes)),
        classes
    )

    ax.set_xlabel(
        "Predicted label",
        fontweight="bold"
    )

    ax.set_ylabel(
        "True label",
        fontweight="bold"
    )

    ax.set_title(
        f"{split.title()} Confusion Matrix",
        fontweight="bold"
    )

    threshold = (
        cm.max() / 2
        if cm.size
        else 0
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                color=(
                    "white"
                    if cm[i, j] > threshold
                    else "black"
                ),
                fontweight="bold"
            )

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / f"{split}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

def plot_roc(
    y_true,
    probs,
    classes,
    split
):
    yb = label_binarize(
        y_true,
        classes=np.arange(len(classes))
    )

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    colors = plt.cm.hsv(
        np.linspace(
            0,
            0.85,
            len(classes)
        )
    )

    for i, (cls, color) in enumerate(
        zip(
            classes,
            colors
        )
    ):
        try:
            fpr, tpr, _ = roc_curve(
                yb[:, i],
                probs[:, i]
            )

            roc_auc = auc(
                fpr,
                tpr
            )

            ax.plot(
                fpr,
                tpr,
                color=color,
                linewidth=2.5,
                label=f"{cls} (AUC={roc_auc:.3f})"
            )
        except Exception:
            pass

    ax.plot(
        [0, 1],
        [0, 1],
        "--",
        color="black"
    )

    ax.set_xlabel(
        "False Positive Rate",
        fontweight="bold"
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontweight="bold"
    )

    ax.set_title(
        f"{split.title()} ROC-AUC",
        fontweight="bold"
    )

    ax.legend(
        loc="lower right"
    )

    ax.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / f"{split}_roc_auc.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

def plot_metric_box(
    per_class_df,
    split
):
    cols = [
        "accuracy_ovr",
        "precision",
        "recall_sensitivity",
        "specificity",
        "f1_score",
        "auc_ovr"
    ]

    names = [
        "Accuracy",
        "Precision",
        "Recall/\nSensitivity",
        "Specificity",
        "F1",
        "AUC"
    ]

    data = [
        per_class_df[c]
        .dropna()
        .values
        for c in cols
    ]

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    bp = ax.boxplot(
        data,
        labels=names,
        patch_artist=True,
        showmeans=True
    )

    colors = plt.cm.hsv(
        np.linspace(
            0,
            0.85,
            len(bp["boxes"])
        )
    )

    for patch, color in zip(
        bp["boxes"],
        colors
    ):
        patch.set_facecolor(
            color
        )

        patch.set_alpha(
            0.65
        )

    ax.set_ylim(
        0,
        1.05
    )

    ax.set_ylabel(
        "Metric value",
        fontweight="bold"
    )

    ax.set_title(
        f"{split.title()} Metric Box Plot",
        fontweight="bold"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / f"{split}_metric_boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

def plot_confidence_box(
    y_true,
    probs,
    classes,
    split
):
    confidence = probs.max(
        axis=1
    )

    data = [
        confidence[
            y_true == i
        ]
        for i in range(
            len(classes)
        )
    ]

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    bp = ax.boxplot(
        data,
        labels=classes,
        patch_artist=True,
        showmeans=True
    )

    colors = plt.cm.hsv(
        np.linspace(
            0,
            0.85,
            len(bp["boxes"])
        )
    )

    for patch, color in zip(
        bp["boxes"],
        colors
    ):
        patch.set_facecolor(
            color
        )

        patch.set_alpha(
            0.65
        )

    ax.set_ylim(
        0,
        1.05
    )

    ax.set_ylabel(
        "Prediction confidence",
        fontweight="bold"
    )

    ax.set_title(
        f"{split.title()} Confidence Box Plot",
        fontweight="bold"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / f"{split}_confidence_boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# ============================================================
# 12. XAI
# ============================================================
def xai_transform(image_rgb):
    transform = transforms.Compose([
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        ),
    ])

    processed = paper_preprocess(
        image_rgb
    )

    return transform(
        Image.fromarray(
            processed
        )
    )

def normalize_map(x):
    x = np.asarray(
        x,
        dtype=np.float32
    )

    x = np.nan_to_num(
        x
    )

    x = x - x.min()

    if x.max() > 0:
        x = x / x.max()

    return x

def heatmap_overlay(
    display_rgb,
    heatmap,
    alpha=0.45
):
    heatmap = normalize_map(
        heatmap
    )

    heatmap = cv2.resize(
        heatmap.astype(
            np.float32
        ),
        (
            display_rgb.shape[1],
            display_rgb.shape[0]
        )
    )

    colored = cv2.applyColorMap(
        np.uint8(
            heatmap * 255
        ),
        cv2.COLORMAP_JET
    )

    colored = cv2.cvtColor(
        colored,
        cv2.COLOR_BGR2RGB
    )

    return cv2.addWeighted(
        display_rgb.astype(
            np.uint8
        ),
        1 - alpha,
        colored.astype(
            np.uint8
        ),
        alpha,
        0
    )

def input_gradient_maps(
    model,
    image_tensor,
    class_index
):
    x = image_tensor.unsqueeze(
        0
    ).to(
        DEVICE
    )

    x.requires_grad_(
        True
    )

    model.eval()
    model.zero_grad(
        set_to_none=True
    )

    logits = model(
        x
    )

    score = logits[
        0,
        class_index
    ]

    score.backward()

    grad = x.grad.detach()[0]

    saliency = (
        grad.abs()
        .max(dim=0)
        .values
        .cpu()
        .numpy()
    )

    # Coarse region attribution.
    energy = (
        grad.abs()
        .mean(dim=0, keepdim=True)
        .unsqueeze(0)
    )

    pooled = torch.nn.functional.avg_pool2d(
        energy,
        kernel_size=16,
        stride=16
    )

    coarse = (
        pooled
        .squeeze()
        .cpu()
        .numpy()
    )

    return (
        normalize_map(saliency),
        normalize_map(coarse)
    )

def generate_xai(
    model,
    test_df,
    classes,
    class_to_idx
):
    for sub in [
        "saliency",
        "swin_region_attribution",
        "lime",
        "combined"
    ]:
        (
            XAI_DIR / sub
        ).mkdir(
            parents=True,
            exist_ok=True
        )

    def lime_predict(images):
        tensors = []

        for img in images:
            tensors.append(
                xai_transform(
                    np.asarray(
                        img,
                        dtype=np.uint8
                    )
                )
            )

        batch = torch.stack(
            tensors
        ).to(
            DEVICE
        )

        with torch.no_grad():
            probs = torch.softmax(
                model(batch),
                dim=1
            )

        return probs.cpu().numpy()

    lime_exp = lime_image.LimeImageExplainer(
        random_state=SEED
    )

    records = []

    for i, (_, row) in enumerate(
        test_df.head(
            N_XAI
        ).iterrows(),
        start=1
    ):
        print(
            f"Generating XAI {i}/{min(N_XAI, len(test_df))}: "
            f"{row.filepath}"
        )

        rgb = read_rgb(
            row.filepath
        )

        display = cv2.resize(
            rgb,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )

        tensor = xai_transform(
            rgb
        )

        with torch.no_grad():
            probs = torch.softmax(
                model(
                    tensor.unsqueeze(
                        0
                    ).to(
                        DEVICE
                    )
                ),
                dim=1
            )[0].cpu().numpy()

        pred = int(
            np.argmax(
                probs
            )
        )

        true_idx = class_to_idx[
            row.label
        ]

        saliency, region = input_gradient_maps(
            model,
            tensor,
            pred
        )

        saliency_img = heatmap_overlay(
            display,
            saliency
        )

        region_img = heatmap_overlay(
            display,
            region
        )

        stem = (
            f"Patient_{i:02d}"
            f"_true_{classes[true_idx]}"
            f"_pred_{classes[pred]}"
        )

        cv2.imwrite(
            str(
                XAI_DIR
                / "saliency"
                / f"{stem}.png"
            ),
            cv2.cvtColor(
                saliency_img,
                cv2.COLOR_RGB2BGR
            )
        )

        cv2.imwrite(
            str(
                XAI_DIR
                / "swin_region_attribution"
                / f"{stem}.png"
            ),
            cv2.cvtColor(
                region_img,
                cv2.COLOR_RGB2BGR
            )
        )

        lime_img = display.copy()

        try:
            explanation = lime_exp.explain_instance(
                display.astype(
                    float
                ),
                classifier_fn=lime_predict,
                top_labels=len(classes),
                hide_color=0,
                num_samples=LIME_NUM_SAMPLES,
                random_seed=SEED
            )

            temp, mask = explanation.get_image_and_mask(
                pred,
                positive_only=True,
                num_features=10,
                hide_rest=False
            )

            lime_img = np.uint8(
                np.clip(
                    mark_boundaries(
                        np.clip(
                            temp,
                            0,
                            255
                        ) / 255.0,
                        mask
                    ),
                    0,
                    1
                ) * 255
            )

            cv2.imwrite(
                str(
                    XAI_DIR
                    / "lime"
                    / f"{stem}.png"
                ),
                cv2.cvtColor(
                    lime_img,
                    cv2.COLOR_RGB2BGR
                )
            )

        except Exception as e:
            print(
                "LIME failed:",
                e
            )

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(18, 5)
        )

        for ax, image, title in zip(
            axes,
            [
                display,
                saliency_img,
                region_img,
                lime_img
            ],
            [
                "Original",
                "Saliency",
                "Swin Region Attribution",
                "LIME"
            ]
        ):
            ax.imshow(
                image
            )

            ax.set_title(
                title,
                fontweight="bold"
            )

            ax.axis(
                "off"
            )

        fig.suptitle(
            f"True: {classes[true_idx]} | "
            f"Predicted: {classes[pred]} "
            f"({probs[pred]:.4f})",
            fontweight="bold"
        )

        plt.tight_layout(
            rect=[0, 0, 1, 0.92]
        )

        plt.savefig(
            XAI_DIR
            / "combined"
            / f"{stem}.png",
            dpi=250,
            bbox_inches="tight"
        )

        plt.close()

        rec = {
            "patient_number": i,
            "filepath": row.filepath,
            "true_class": classes[true_idx],
            "predicted_class": classes[pred],
            "confidence": float(
                probs[pred]
            ),
            "correct": int(
                true_idx == pred
            ),
        }

        for j, cls in enumerate(classes):
            rec[
                f"prob_{cls}"
            ] = float(
                probs[j]
            )

        records.append(
            rec
        )

    pd.DataFrame(
        records
    ).to_csv(
        XAI_DIR / "first_20_test_xai_predictions.csv",
        index=False
    )

# ============================================================
# 13. COMPLEXITY
# ============================================================
def save_complexity(
    model,
    training_time
):
    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    non_trainable = (
        total - trainable
    )

    complexity = {
        "model": "Pretrained Swin Transformer Tiny",
        "pretraining": "ImageNet-1K",
        "input_size": [IMG_SIZE, IMG_SIZE],
        "total_parameters": int(total),
        "trainable_parameters_after_final_finetuning": int(trainable),
        "non_trainable_parameters_after_final_finetuning": int(non_trainable),
        "training_time_seconds": float(training_time),
        "training_time_minutes": float(
            training_time / 60.0
        ),
    }

    with open(
        METRICS_DIR / "model_complexity_and_time.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            complexity,
            f,
            indent=2
        )

    pd.DataFrame(
        [complexity]
    ).to_csv(
        METRICS_DIR / "model_complexity_and_time.csv",
        index=False
    )

# ============================================================
# 14. ZIP
# ============================================================
def zip_and_download():
    zip_path = (
        WORK_DIR
        / "Pretrained_Swin_Tiny_Liver_Tumor_Results.zip"
    )

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as zf:

        for f in RESULTS_DIR.rglob("*"):
            if f.is_file():
                zf.write(
                    f,
                    arcname=f.relative_to(
                        RESULTS_DIR.parent
                    )
                )

    print(
        "ZIP created:",
        zip_path
    )

    if in_colab():
        from google.colab import files

        files.download(
            str(zip_path)
        )
    else:
        print(
            "Download manually:",
            zip_path.resolve()
        )

# ============================================================
# 15. MAIN
# ============================================================
def main():
    total_start = time.perf_counter()

    kaggle_sources = download_data()

    df = collect_images()

    classes = [
        c
        for c in TARGET_CLASS_ORDER
        if c in df["label"].unique()
    ]

    class_to_idx = {
        c: i
        for i, c
        in enumerate(classes)
    }

    (
        RESULTS_DIR
        / "class_mapping.json"
    ).write_text(
        json.dumps(
            class_to_idx,
            indent=2
        ),
        encoding="utf-8"
    )

    train_df, val_df, test_df = create_splits(
        df
    )

    save_before_after(
        train_df
    )

    train_ds = LiverDataset(
        train_df,
        class_to_idx,
        training=True
    )

    train_eval_ds = LiverDataset(
        train_df,
        class_to_idx,
        training=False
    )

    val_ds = LiverDataset(
        val_df,
        class_to_idx,
        training=False
    )

    test_ds = LiverDataset(
        test_df,
        class_to_idx,
        training=False
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    train_eval_loader = DataLoader(
        train_eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    y_train = np.asarray([
        class_to_idx[
            label
        ]
        for label
        in train_df.label
    ])

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(
            len(classes)
        ),
        y=y_train
    )

    print(
        "Class weights:",
        {
            classes[i]: float(
                class_weights[i]
            )
            for i in range(
                len(classes)
            )
        }
    )

    print(
        "Loading pretrained Swin Transformer Tiny "
        "ImageNet-1K weights..."
    )

    model = build_model(
        len(classes)
    ).to(
        DEVICE
    )

    print(
        "Total parameters:",
        f"{sum(p.numel() for p in model.parameters()):,}"
    )

    training_start = time.perf_counter()

    history = fit_model(
        model,
        train_loader,
        val_loader,
        test_loader,
        classes,
        class_weights
    )

    training_time = (
        time.perf_counter()
        - training_start
    )

    plot_history(
        history
    )

    final_metrics = []

    for split, loader in [
        (
            "training",
            train_eval_loader
        ),
        (
            "validation",
            val_loader
        ),
        (
            "test",
            test_loader
        ),
    ]:
        overall, y, pred, probs, per_class = evaluate_split(
            model,
            loader,
            split,
            classes
        )

        final_metrics.append(
            overall
        )

        plot_confusion(
            y,
            pred,
            classes,
            split
        )

        plot_roc(
            y,
            probs,
            classes,
            split
        )

        plot_metric_box(
            per_class,
            split
        )

        plot_confidence_box(
            y,
            probs,
            classes,
            split
        )

    final_df = pd.DataFrame(
        final_metrics
    )

    final_df.to_csv(
        METRICS_DIR
        / "all_split_overall_metrics.csv",
        index=False
    )

    print(
        "\nFINAL METRICS\n"
    )

    print(
        final_df.to_string(
            index=False
        )
    )

    torch.save(
        {
            "model_state_dict":
                model.state_dict(),
            "class_to_idx":
                class_to_idx,
            "classes":
                classes,
            "architecture":
                "swin_t",
            "weights":
                "IMAGENET1K_V1",
        },
        MODEL_DIR
        / "final_pretrained_swin_tiny.pth"
    )

    save_complexity(
        model,
        training_time
    )

    generate_xai(
        model,
        test_df,
        classes,
        class_to_idx
    )

    metadata = {
        "model":
            "Pretrained Swin Transformer Tiny",
        "framework":
            "PyTorch/Torchvision",
        "pretrained_weights":
            "Swin_T_Weights.IMAGENET1K_V1",
        "classes":
            classes,
        "class_mapping":
            class_to_idx,
        "kaggle_sources":
            kaggle_sources,
        "image_size":
            IMG_SIZE,
        "preprocessing": {
            "bilateral_filter":
                USE_BILATERAL_FILTER,
            "clahe":
                USE_CLAHE,
            "imagenet_normalization":
                True,
        },
        "training": {
            "head_epochs":
                HEAD_EPOCHS,
            "fine_tune_epochs":
                FINE_TUNE_EPOCHS,
            "fine_tuned_region":
                "final Swin stage + norm + classifier head",
        },
        "xai": [
            "Input-gradient saliency",
            "Swin coarse region attribution",
            "LIME",
        ],
        "total_pipeline_time_seconds":
            float(
                time.perf_counter()
                - total_start
            ),
    }

    with open(
        RESULTS_DIR
        / "run_metadata.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )

    zip_and_download()

if __name__ == "__main__":
    main()


PyTorch version: 2.11.0+cu128
Device: cuda
Kaggle credentials found: /root/.kaggle/kaggle.json
[CMD] kaggle kernels pull ahmedhamza1996/liver-tumor-classification -p /content/liver_swin_tiny_work/kaggle_kernel_metadata -m
Source code and metadata downloaded to /content/liver_swin_tiny_work/kaggle_kernel_metadata

Discovered Kaggle dataset sources: ['ahmedhamza1996/dataset']
[CMD] kaggle datasets download -d ahmedhamza1996/dataset -p /content/liver_swin_tiny_work/dataset/ahmedhamza1996__dataset --unzip
ing an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)
Dataset URL: https://www.kaggle.com/datasets/ahmedhamza1996/dataset
License(s): unknown

  0%|          | 0.00/241M [00:00<?, ?B/s]
  3%|▎         | 7.00M/241M [00:00<00:04, 60.0MB/s]
  7%|▋         | 17.0M/241M [00:00<00:02, 81.2MB/s]
 12%|█▏        | 28.0M/241M [00:00<00:02, 95.3MB/s]
 18%|█▊        | 43.0M/241M [00:00<00:01, 113MB/s] 
 22%|██▏       | 54.0M/241M [00:00<00:01, 11

100%|██████████| 108M/108M [00:00<00:00, 179MB/s]


Total parameters: 27,521,661
Epoch 01/30 | train_acc=0.5879 | val_acc=0.8014 | test_acc=0.8242 | val_loss=0.6014
Epoch 02/30 | train_acc=0.7741 | val_acc=0.6644 | test_acc=0.7582 | val_loss=0.4695
Epoch 03/30 | train_acc=0.8448 | val_acc=0.7603 | test_acc=0.8407 | val_loss=0.3718
Epoch 04/30 | train_acc=0.8914 | val_acc=0.8151 | test_acc=0.8901 | val_loss=0.3348
Epoch 05/30 | train_acc=0.9034 | val_acc=0.8562 | test_acc=0.9066 | val_loss=0.2923
Epoch 06/30 | train_acc=0.9000 | val_acc=0.8973 | test_acc=0.9286 | val_loss=0.2802
Epoch 07/30 | train_acc=0.9172 | val_acc=0.8973 | test_acc=0.9286 | val_loss=0.2659
Epoch 08/30 | train_acc=0.9172 | val_acc=0.9110 | test_acc=0.9451 | val_loss=0.2291
Epoch 09/30 | train_acc=0.9190 | val_acc=0.9247 | test_acc=0.9451 | val_loss=0.2105
Epoch 10/30 | train_acc=0.9241 | val_acc=0.9452 | test_acc=0.9505 | val_loss=0.1918
Epoch 11/30 | train_acc=0.9345 | val_acc=0.9315 | test_acc=0.9505 | val_loss=0.1378
Epoch 12/30 | train_acc=0.9621 | val_acc=0.9658

  0%|          | 0/400 [00:00<?, ?it/s]

Generating XAI 2/20: /content/liver_swin_tiny_work/dataset/ahmedhamza1996__dataset/output/benign/(370).PNG


...

Generating XAI 20/20: /content/liver_swin_tiny_work/dataset/ahmedhamza1996__dataset/output/malignant/(55).PNG


  0%|          | 0/400 [00:00<?, ?it/s]

ZIP created: /content/liver_swin_tiny_work/Pretrained_Swin_Tiny_Liver_Tumor_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>